In [30]:
!pip install langchain_community langchain_text_splitters langchain_openai langchain_chroma

In [31]:
import os

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [32]:
from getpass import getpass

openai_api_key = getpass("Enter your OpenAI API key: ")

os.environ["OPENAI_API_KEY"] = openai_api_key

Enter your OpenAI API key: ··········


In [35]:
jeju_content = """
[제주도 맛집 & 여행 가이드]
1. 흑돼지 거리: 제주시 건입동에 위치. 저녁 7시 이후에는 웨이팅이 1시간 이상 발생할 수 있음.
   추천 메뉴는 흑돼지 오겹살(200g 2만원). 멜젓에 찍어 먹는 것이 특징.

2. 우도 땅콩 아이스크림: 우도 검멀레 해변 앞이 원조. 가격은 5,000원.
   고소한 땅콩 가루가 듬뿍 뿌려져 있음. 배 시간 때문에 오후 4시 이전에 가는 것을 추천.

3. 성산일출봉: 입장료 5,000원. 왕복 소요 시간 50분.
   매월 첫째 주 월요일은 휴관. 새벽 일출을 보려면 전날 근처 숙소 예약 필수.
"""

with open("jeju_content.txt", "w", encoding="utf-8") as f:
    f.write(jeju_content)

print("[준비 완료] 제주도 맛집 & 여행 가이드 문서 생성")

[준비 완료] 제주도 맛집 & 여행 가이드 문서 생성


In [36]:
print("\n 문서를 불러옵니다...")

loader = TextLoader("jeju_content.txt", encoding="utf-8")
docs = loader.load()

print(f"    => 문서 길이: {len(docs[0].page_content)}자")


 문서를 불러옵니다...
    => 문서 길이: 315자


In [37]:
print("\n 문서를 작은 조각(Chunk)으로 나눕니다...")

text_splitter = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)
splits = text_splitter.split_documents(docs)

print(f"    => 총 {len(splits)}개의 조각(Chunk)으로 분할")


 문서를 작은 조각(Chunk)으로 나눕니다...
    => 총 4개의 조각(Chunk)으로 분할


In [41]:
print("\n 문서를 벡터로 변환하여 저장합니다...")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=OpenAIEmbeddings(model="text-embedding-3-small")
    )

print("    => Chroma DB에 저장 완료")


 문서를 벡터로 변환하여 저장합니다...
    => Chroma DB에 저장 완료


In [43]:
print("\n RAG 체인을 생성합니다...")

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


 RAG 체인을 생성합니다...


In [77]:
template = """
You are a 10-year experienced Jeju Island local guide expert.
MISSION: Provide accurate and practical answers based on the given information.

STRICT RULES:
1. Refer ONLY to the [Guide Information] below for your answers.
2. If the information not in the guide, say only "This information is not in the guide."
3. If the information is in the guide, provide the answer as a 10-year experienced Jeju Island local guide expert. Explain thoroughly and correctly.
4. Do not speculate about uncertain information.
5. Provide clear and actionable answers.

[Guide Information]
{context}

[Question]: {query}

ANSWER FORMAT: Use a friendly and professional tone, use bullet points when necessary.
"""
prompt = PromptTemplate.from_template(template)

In [78]:
rag_chain = (
    {"context": retriever, "query": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [79]:
query = "흑돼지 먹으려면 어디로 가야 해? 그리고 성산일출봉 입장료는 얼마야?"

print(f"\n 질문: {query}")
print("AI 답변 생성 중...\n")

response = rag_chain.invoke(query)

print(response)


 질문: 흑돼지 먹으려면 어디로 가야 해? 그리고 성산일출봉 입장료는 얼마야?
AI 답변 생성 중...

- **흑돼지 먹으려면 어디로 가야 해?**
  - 추천 메뉴는 흑돼지 오겹살로, 가격은 200g에 20,000원입니다. 멜젓에 찍어 먹는 것이 특징입니다. 특정 식당에 대한 정보는 제공되지 않았으므로, 흑돼지를 제공하는 식당을 찾아보시는 것이 좋습니다.

- **성산일출봉 입장료는 얼마야?**
  - 성산일출봉의 입장료는 5,000원입니다. 왕복 소요 시간은 약 50분이니, 방문 계획 시 시간을 고려하시기 바랍니다.
